# Train a named prototype method
Supply external settings and two class-disjoint data roots. The original trainer calls its validation argument `test_loader`; here it receives validation data. Final test data are not used for checkpoint selection. No settings or trained results are included.


In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from amgpn.config import require as cfg, PROJECT_ROOT
from amgpn.data.episodes import UAVDataset, MetaDataset
from amgpn.experiments import create_experiment

EXPERIMENT_ID = "amgpn"  # Choose "pn", "gpn", or "amgpn" explicitly.
components = create_experiment(
    EXPERIMENT_ID, device=cfg("workflow.device"),
    model_kwargs=cfg("workflow.training.model_kwargs"),
    trainer_kwargs=cfg("workflow.training.trainer_kwargs"),
)
train_data = UAVDataset(cfg("workflow.training.data_root"), seed=cfg("workflow.dataset_seed"),
                        is_train=True, train_ratio=1.0, max_sample_count=cfg("workflow.max_sample_count"))
validation_data = UAVDataset(cfg("workflow.training.validation_root"), seed=cfg("workflow.dataset_seed"),
                             is_train=True, train_ratio=1.0, max_sample_count=cfg("workflow.max_sample_count"))
train_labels = {label for _, label in train_data}
validation_labels = {label for _, label in validation_data}
assert train_labels.isdisjoint(validation_labels), "Training and validation roots must contain disjoint classes."

n_way, k_shot, q_query = cfg("workflow.n_way"), cfg("workflow.training.k_shot"), cfg("workflow.q_query")
train_episodes = MetaDataset(train_data, n_way, k_shot, q_query,
                            num_tasks=cfg("workflow.training.num_tasks"), seed=cfg("workflow.episode_seed"))
validation_episodes = MetaDataset(validation_data, n_way, k_shot, q_query,
                                 num_tasks=cfg("workflow.training.validation_tasks"), seed=cfg("workflow.episode_seed"))
train_loader = DataLoader(train_episodes, batch_size=cfg("workflow.training.batch_size"), shuffle=True)
validation_loader = DataLoader(validation_episodes, batch_size=1, shuffle=False)
checkpoint = Path(cfg("workflow.training.checkpoint_path")).expanduser().resolve()
assert not checkpoint.is_relative_to(PROJECT_ROOT), "Keep checkpoints outside the repository."
checkpoint.parent.mkdir(parents=True, exist_ok=True)
components.trainer.train(train_loader, validation_loader,
                         epochs=cfg("workflow.training.epochs"), n_ways=n_way, save_path=str(checkpoint))